In [17]:
import dspy

# Konfiguration des lokalen Sprachmodells
local_llm = dspy.LM(
    "openai/Qwen3-VL-8B-Instruct-Q4_K_M.gguf", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=0.1,
    cache=False
)

dspy.configure(lm=local_llm)

In [18]:
def sentiment_accuracy(gold, pred, trace=None):
    """
    Berechnet die Genauigkeit für die Sentiment-Klassifizierung.
    Gibt True zurück, wenn die Vorhersage mit dem Gold-Label übereinstimmt, andernfalls False.
    
    Args:
        gold (dspy.Example): Das Beispiel mit der korrekten Antwort.
        pred (dspy.Prediction): Die Vorhersage des Modells.
        trace (optional): Der Ausführungstrace. Wird hier nicht verwendet.
        
    Returns:
        bool: True bei Übereinstimmung, andernfalls False.
    """
    # Zugriff auf die relevanten Felder in den gold- und pred-Objekten
    gold_sentiment = gold.sentiment
    predicted_sentiment = pred.sentiment

    # Debugging only
    # print (gold_sentiment, predicted_sentiment)
    
    # Normalisierung und Vergleich der Werte (Groß-/Kleinschreibung ignorieren)
    return gold_sentiment.lower() == predicted_sentiment.lower()

In [19]:
class SentimentSignature(dspy.Signature):
    """Klassifiziert den Sentiment eines gegebenen Textes als positiv oder negativ."""
    input = dspy.InputField(desc="Der zu klassifizierende Text.")
    sentiment = dspy.OutputField(desc="Das Ergebnis der Klassifizierung: positiv oder negativ.")

class SimpleClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.Predict(SentimentSignature)

    def forward(self, input):
        return self.predictor(input=input)

In [20]:
from datasets import load_dataset

# Datensatz laden – hier der IMDb Sentiment-Datensatz
dataset = load_dataset("imdb", split="train") 
dataset = dataset.shuffle(seed=42)
dataset = dataset.select(range(500))

In [21]:
from collections import Counter

# Check label distribution
label_counts = Counter(dataset["label"])

total = sum(label_counts.values())
for label, count in label_counts.items():
    print(f"Label {label}: {count} ({count/total:.2%})")


Label 1: 246 (49.20%)
Label 0: 254 (50.80%)


In [22]:
# DSPy-Examples erzeugen
examples = [
    dspy.Example(
        input=text["text"],
        sentiment="positiv" if text["label"] == 1 else "negativ"
    ).with_inputs("input")  # Definiert, welches Feld als Eingabe genutzt wird
    for text in dataset
]

# Testausgabe
print(examples[0])
print(f"{len(examples)} DSPy Examples geladen.")

Example({'input': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'sentiment': 'positiv'}) (input_keys={'input'})
500 DSPy Examples geladen.


In [23]:
# Aufteilung des Datensatzes
split_index = int(len(examples) * 0.8)  # 80/20-Split

trainset = examples[:split_index]
devset = examples[split_index:]

# Anzeige der Größe der erstellten Sets
print(f"Anzahl der Beispiele im Trainingsset: {len(trainset)}")
print(f"Anzahl der Beispiele im Evaluationsset: {len(devset)}")

Anzahl der Beispiele im Trainingsset: 400
Anzahl der Beispiele im Evaluationsset: 100


In [24]:
from dspy.teleprompt import BootstrapFewShot

# Metrik für die Optimierung
def metric(gold, pred, trace=None):
    return gold.sentiment.lower() == pred.sentiment.lower()

# Konfiguration und Ausführung des Optimizers
optimizer = BootstrapFewShot(metric=metric, max_bootstrapped_demos=10, max_rounds=5)
optimized_classifier = optimizer.compile(SimpleClassifier(), trainset=trainset)

  3%|██▏                                                                              | 11/400 [00:12<07:09,  1.10s/it]

Bootstrapped 10 full traces after 11 examples for up to 5 rounds, amounting to 15 attempts.


In [25]:
from dspy.evaluate import Evaluate

# Instanziierung des Evaluators
evaluator = Evaluate(devset=devset, num_threads=1, display_progress=True)

# Ausführung der Evaluation mit der benutzerdefinierten Metrik
evaluation_score = evaluator(optimized_classifier, metric=sentiment_accuracy)

Average Metric: 92.00 / 100 (92.0%): 100%|███████████████████████████████████████████| 100/100 [01:06<00:00,  1.51it/s]

2025/11/13 14:06:20 INFO dspy.evaluate.evaluate: Average Metric: 92 / 100 (92.0%)


In [26]:
# Ausführung mit dem Testtext (Ein Beispiel aus dem imdb Datensatz)
test_text = "Piece of subtle art. Maybe a masterpiece. Doubtlessly a special story about the ambiguity of existence. Tale in Kafka style about impossibility of victory or surviving in a perpetual strange world. The life is, in this film, only exercise of adaptation. Lesson about limits and original sin, about the frailty of innocence and error of his ways.<br /><br />Leopold Kessle is another Joseph K. Images of Trial and same ambiguous woman. And Europa is symbol of basic crisis who has many aspects like chimeric wars or unavailing search of truth/essence/golden age.<br /><br />Methaphor or parable, the movie is history of disappointed's evolution. War, peace, business or lie are only details of gelatin-time. Hypocrisy is a mask. Love- a convention. The sacrifice- only method to hope understanding a painful reality."
optimized_classifier(input=test_text)

Prediction(
    sentiment='positiv'
)

In [27]:
unoptimized_classifier = SimpleClassifier()

# Instanziierung des Evaluators
evaluator = Evaluate(devset=devset, num_threads=1, display_progress=True)

evaluation_score = evaluator(unoptimized_classifier, metric=sentiment_accuracy)

Average Metric: 94.00 / 100 (94.0%): 100%|███████████████████████████████████████████| 100/100 [01:13<00:00,  1.37it/s]

2025/11/13 14:07:44 INFO dspy.evaluate.evaluate: Average Metric: 94 / 100 (94.0%)


In [28]:
# Ausführung mit dem Testtext (Ein Beispiel aus dem imdb Datensatz)
test_text = "Piece of subtle art. Maybe a masterpiece. Doubtlessly a special story about the ambiguity of existence. Tale in Kafka style about impossibility of victory or surviving in a perpetual strange world. The life is, in this film, only exercise of adaptation. Lesson about limits and original sin, about the frailty of innocence and error of his ways.<br /><br />Leopold Kessle is another Joseph K. Images of Trial and same ambiguous woman. And Europa is symbol of basic crisis who has many aspects like chimeric wars or unavailing search of truth/essence/golden age.<br /><br />Methaphor or parable, the movie is history of disappointed's evolution. War, peace, business or lie are only details of gelatin-time. Hypocrisy is a mask. Love- a convention. The sacrifice- only method to hope understanding a painful reality."
unoptimized_classifier(input=test_text)

Prediction(
    sentiment='positiv'
)

In [16]:
# Anzeige des letzten generierten Prompts
local_llm.inspect_history(n=1)





[2025-11-13T13:50:54.333653]

System message:

Your input fields are:
1. `input` (str): Der zu klassifizierende Text.
Your output fields are:
1. `sentiment` (str): Das Ergebnis der Klassifizierung: positiv oder negativ.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## input ## ]]
{input}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Klassifiziert den Sentiment eines gegebenen Textes als positiv oder negativ.


User message:

[[ ## input ## ]]
Piece of subtle art. Maybe a masterpiece. Doubtlessly a special story about the ambiguity of existence. Tale in Kafka style about impossibility of victory or surviving in a perpetual strange world. The life is, in this film, only exercise of adaptation. Lesson about limits and original sin, about the frailty of innocence and error of his ways.<br /><br />Leopold Kessle is another Joseph K. Images of Trial and same ambig